
# BiLSTM vs. MolFormer — Encoder Comparison

This notebook compares two SMILES-based molecular encoders on the BACE-1 bioactivity prediction task:

- **BiLSTM:** A trainable bidirectional LSTM encoder.
- **MolFormer:** A pretrained transformer encoder with frozen weights and a trainable regression head.




Both encoders are compared using the same BACE-1 dataset, target variable, and scaffold split, verified through dataset and split hashes. The comparison uses recorded validation results from previously completed hyperparameter sweeps. Configurations are ranked by mean validation RMSE across three random seeds. Test-set evaluation is outside the scope of this notebook.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# locate the AI directory regardless of where Jupyter starts
cwd = Path.cwd().resolve()

if (cwd / "ai" / "outputs").is_dir():
    ROOT = cwd / "ai"
elif (cwd / "outputs").is_dir():
    ROOT = cwd
elif (cwd.parent / "outputs").is_dir():
    ROOT = cwd.parent
else:
    raise FileNotFoundError("Could not locate ai/outputs")

bilstm = pd.read_csv(
    ROOT / "outputs/encoders/BiLSTM/sweep/3-finalists.csv"
)

molformer = pd.read_csv(
    ROOT / "outputs/encoders/SMILES/1-head.csv"
)

print("BiLSTM runs:", len(bilstm))
print("MolFormer runs:", len(molformer))

In [ ]:

# verify that both encoders used the same dataset and split
for column in ("data_sha256", "split_sha256"):
    if column not in bilstm.columns or column not in molformer.columns:
        raise ValueError(f"Missing {column} in encoder results")

    if bilstm[column].isna().any() or molformer[column].isna().any():
        raise ValueError(f"Missing values in {column}")

    bilstm_values = bilstm[column].unique()
    molformer_values = molformer[column].unique()

    if len(bilstm_values) != 1 or len(molformer_values) != 1:
        raise ValueError(f"Inconsistent {column} values")

    if bilstm_values[0] != molformer_values[0]:
        raise ValueError(
            f"BiLSTM and MolFormer have different {column} values"
        )

print("Dataset hashes match!")
print("Split hashes match!")

In [ ]:

import json

required_seeds = {0, 1, 2}

# complete intended BiLSTM finalist configurations
bilstm_base = {
    "EMBEDDING_DIM": 128,
    "HIDDEN_DIM": 128,
    "NUM_LAYERS": 2,
    "DROPOUT": 0.2,
    "LR": 0.001,
    "WEIGHT_DECAY": 0.0001,
    "BATCH_SIZE": 64,
    "EPOCHS": 40,
    "PATIENCE": 8,
    "GRAD_CLIP": 1.0,
    "OPT": "adamw",
}

bilstm_expected = [
    bilstm_base,
    bilstm_base | {"EMBEDDING_DIM": 64, "HIDDEN_DIM": 64},
    bilstm_base | {
        "EMBEDDING_DIM": 64,
        "HIDDEN_DIM": 256,
        "NUM_LAYERS": 1,
        "BATCH_SIZE": 128,
    },
]

def validate_runs(frame, expected, name):
    expected_ids = {json.dumps(cfg, sort_keys=True) for cfg in expected}
    result = frame.copy()

    result["_config_id"] = result["cfg"].apply(
        lambda cfg: json.dumps(json.loads(cfg), sort_keys=True)
    )

    # exclude custom overrides and unrelated configurations
    result = result[result["_config_id"].isin(expected_ids)].copy()

    if set(result["_config_id"]) != expected_ids:
        raise ValueError(f"{name}: missing expected configurations")

    for cfg, group in result.groupby("_config_id"):
        seeds = group["seed"].tolist()
        if len(seeds) != 3 or set(seeds) != required_seeds:
            raise ValueError(
                f"{name}: expected exactly seeds 0, 1, 2 for {cfg}"
            )

    return result

bilstm = validate_runs(bilstm, bilstm_expected, "BiLSTM")

bilstm_summary = (
    bilstm.groupby("_config_id")
    .agg(
        mean_RMSE=("val_RMSE", "mean"),
        std_RMSE=("val_RMSE", "std"),
        mean_R2=("val_R2", "mean"),
        seeds=("seed", "nunique"),
    )
    .sort_values("mean_RMSE")
)

# complete default MolFormer 1-head configurations
molformer_base = {
    "MODEL_NAME": "ibm-research/MoLFormer-XL-both-10pct",
    "PRETRAINED_REVISION": "361063d0ad524ef77cf39b08469f6be770dc550f",
    "FREEZE": True,
    "MAX_LENGTH": 202,
    "ENCODER_BATCH_SIZE": 32,
    "BATCH_SIZE": 64,
    "EPOCHS": 50,
    "LR": 0.001,
    "WEIGHT_DECAY": 0.0,
}

molformer_expected = [
    molformer_base | {"LR": lr}
    for lr in (0.0001, 0.001, 0.01)
]

molformer = validate_runs(molformer, molformer_expected, "MolFormer")
molformer["LR"] = molformer["_config_id"].apply(
    lambda cfg: json.loads(cfg)["LR"]
)

molformer_summary = (
    molformer.groupby("_config_id")
    .agg(
        LR=("LR", "first"),
        mean_RMSE=("val_RMSE", "mean"),
        std_RMSE=("val_RMSE", "std"),
        mean_R2=("val_R2", "mean"),
        seeds=("seed", "nunique"),
    )
    .sort_values("mean_RMSE")
)

print("Best BiLSTM configurations:")
display(bilstm_summary.head(3).round(4))

print("MolFormer learning rate comparison:")
display(molformer_summary.round(4))

In [ ]:
comparison = pd.DataFrame({
    "Encoder": ["BiLSTM", "MolFormer"],
    "Mean Val RMSE": [
        bilstm_summary.iloc[0]["mean_RMSE"],
        molformer_summary.iloc[0]["mean_RMSE"]
    ],
    "Std Val RMSE": [
        bilstm_summary.iloc[0]["std_RMSE"],
        molformer_summary.iloc[0]["std_RMSE"]
    ],
    "Mean Val R²": [
        bilstm_summary.iloc[0]["mean_R2"],
        molformer_summary.iloc[0]["mean_R2"]
    ]
})

display(comparison.round(4))

improvement = (
    (comparison.iloc[0]["Mean Val RMSE"] - comparison.iloc[1]["Mean Val RMSE"])
    / comparison.iloc[0]["Mean Val RMSE"]
    * 100
)

print(f"MolFormer RMSE improvement: {improvement:.2f}%")

In [ ]:
plt.figure(figsize=(7, 4))

plt.bar(
    comparison["Encoder"],
    comparison["Mean Val RMSE"],
    yerr=comparison["Std Val RMSE"],
    capsize=6,
    color=["steelblue", "darkorange"]
)

plt.ylabel("Mean Validation RMSE")
plt.title("BiLSTM vs. MolFormer on BACE-1")
plt.ylim(0, 1.05)
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:

bilstm_rmse = comparison.iloc[0]["Mean Val RMSE"]
molformer_rmse = comparison.iloc[1]["Mean Val RMSE"]

print("Conclusion")
print(
    "BiLSTM: Mean validation RMSE = "
    f"{bilstm_rmse:.4f} ± {comparison.iloc[0]['Std Val RMSE']:.4f}"
)
print(
    "MolFormer: Mean validation RMSE = "
    f"{molformer_rmse:.4f} ± {comparison.iloc[1]['Std Val RMSE']:.4f}"
)

if improvement > 0:
    print(f"MolFormer achieved {improvement:.2f}% lower validation RMSE.")
elif improvement < 0:
    print(f"BiLSTM achieved {-improvement:.2f}% lower validation RMSE.")
else:
    print("Both encoders achieved the same mean validation RMSE.")


## Recorded experiment results

Compared two SMILES-based molecular encoders on the BACE-1 benchmark, selecting each encoder's best hyperparameter configuration based on mean validation RMSE across three random seeds.

- **BiLSTM:** Mean validation RMSE = 0.8859 ± 0.0287, with mean R² = 0.5688.
- **MolFormer:** Mean validation RMSE = 0.8718 ± 0.0010, with mean R² = 0.5827.

MolFormer achieved approximately **1.59% lower validation RMSE** and exhibited less variation across seeds.

These results suggest that frozen pretrained MolFormer embeddings with a linear prediction head provide a modest improvement over the trained BiLSTM encoder under the evaluated configurations.

Note: These numerical results are a record of the completed sweeps, not a dynamically generated conclusion! If the input CSVs are updated or the experiments are rerun, the table, chart, and numbers generated by the code cells above should be consulted for current results.